In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO
from ultralytics.utils.downloads import download

In [ ]:
TRAIN_DATASET_DIR = "./datasets/VisDrone2019-DET-train"
VALIDATION_DATASET_DIR = "./datasets/VisDrone2019-DET-val"
TEST_CHALLENGE_DATASET_DIR = "./datasets/VisDrone2019-DET-test-challenge"
RUNS_DIR = "./runs/detect/runs"

yaml_file = TRAIN_DATASET_DIR + "/data.yaml"

In [ ]:
DATASETS_ROOT = str(Path(TRAIN_DATASET_DIR).resolve().parent)

yaml_content = f"""path: {DATASETS_ROOT}
train: VisDrone2019-DET-train/images
val: VisDrone2019-DET-val/images
nc: 10
names: ['pedestrian', 'people', 'bicycle', 'car', 'van', 'truck', 'tricycle', 'awning-tricycle', 'bus', 'motor']
"""

with open(yaml_file, 'w') as f:
    f.write(yaml_content)

In [ ]:
download("https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11n.pt", dir=".")

In [ ]:
model = YOLO("yolo11n.pt")

results = model.train(
    data=yaml_file,
    epochs=1,
    imgsz=640,
    batch=32,
    device=0,
    patience=5,
    save=True,
    project=RUNS_DIR,
    name="baseline_yolo11n",
    exist_ok=True,
    verbose=True,
    plots=True
)

In [ ]:
results_path = Path(RUNS_DIR) / "baseline_yolo11n"

df = pd.read_csv(str(results_path / "results.csv"))
last_row = df.iloc[-1]

print(f"Epochs: {len(df)}")
print(f"Final Loss: {last_row['train/box_loss']:.4f}")
print(f"Final Val Loss: {last_row['val/box_loss']:.4f}")
print(f"mAP50: {last_row['metrics/mAP50(B)']:.4f}")
print(f"mAP50-95: {last_row['metrics/mAP50-95(B)']:.4f}")
print(f"Precision: {last_row['metrics/precision(B)']:.4f}")
print(f"Recall: {last_row['metrics/recall(B)']:.4f}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Training Dynamics')

train_total = df['train/box_loss'] + df['train/cls_loss'] + df['train/dfl_loss']
val_total   = df['val/box_loss']   + df['val/cls_loss']   + df['val/dfl_loss']

ax = axes[0, 0]
ax.plot(df.index, train_total, 'b-', label='Train Loss', linewidth=2)
ax.plot(df.index, val_total, 'r-', label='Val Loss', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Total Loss')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.plot(df.index, df['train/box_loss'], 'b-', label='Train', linewidth=2)
ax.plot(df.index, df['val/box_loss'], 'r-', label='Val', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Box Loss')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
ax.plot(df.index, df['train/cls_loss'], 'b-', label='Train', linewidth=2)
ax.plot(df.index, df['val/cls_loss'], 'r-', label='Val', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Classification Loss')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
ax.plot(df.index, df['metrics/mAP50(B)'], 'g-', label='mAP50', linewidth=2)
ax.plot(df.index, df['metrics/mAP50-95(B)'], 'orange', label='mAP50-95', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('mAP')
ax.set_title('Mean Average Precision')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Performance Metrics')

ax = axes[0]
ax.plot(df.index, df['metrics/precision(B)'], 'b-', label='Precision', linewidth=2)
ax.plot(df.index, df['metrics/recall(B)'], 'r-', label='Recall', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Score')
ax.set_title('Precision vs Recall')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 1])

ax = axes[1]
metrics_names = ['Precision', 'Recall', 'mAP50', 'mAP50-95']
metrics_values = [
    last_row['metrics/precision(B)'],
    last_row['metrics/recall(B)'],
    last_row['metrics/mAP50(B)'],
    last_row['metrics/mAP50-95(B)']
]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
bars = ax.bar(metrics_names, metrics_values, color=colors, alpha=0.7, edgecolor='black')

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
           f'{height:.3f}',
           ha='center', va='bottom', fontweight='bold')

ax.set_ylabel('Score')
ax.set_title('Final Metrics')
ax.set_ylim([0, 1])
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
best_model_path = results_path / "weights" / "best.pt"
best_model = YOLO(str(best_model_path))

val_results = best_model.val(
    data=yaml_file,
    imgsz=640,
    device=0,
    verbose=False
)

print(f"Validation mAP50: {val_results.box.map50:.4f}")
print(f"Validation mAP50-95: {val_results.box.map:.4f}")
print(f"Validation Precision: {float(val_results.box.p.mean()):.4f}")
print(f"Validation Recall: {float(val_results.box.r.mean()):.4f}")

In [ ]:
# Confusion matrix visualization
best_model.val(data=yaml_file, imgsz=640, device=0, verbose=False, plots=True)